In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [5]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-4",
)

In [ ]:

# SYSTEM_PROMPT = """
# You are an expert in parsing terminal session logs.

# Extract **only the timestamps** where a shell prompt appears inside <system_output>.
# Do not return any text, explanation, or formatting — only timestamps, one per line.

# A shell prompt looks like:
# demo@boxtop:~$
# or any similar prompt indicating that the previous command finished.

# Each timestamp you return **defines an event boundary**.
# Remember: Only return timestamps. Each timestamp defines an event boundary. No extra text or explanation.
# ***Important note: The switch between the user input and system output should not be treated as a boundary.***

# Return format:
# <timestamp>
# """
SYSTEM_PROMPT = """
You are an expert in parsing terminal session logs. Your task is to identify the **earliest** (first-most) event boundary in the provided log.

### Task:
1. Scan the log from top to bottom.
2. Identify the **very first** occurrence of a shell prompt (e.g., `demo@faiserver:~$`) appearing inside a `<system_output>` tag.
3. Extract the timestamp associated with that specific tag.

### Critical Constraints:
- **Source Integrity:** The extracted timestamp **MUST** exist exactly as written within the `timestamp="..."` attribute of a tag in the provided input. Do not invent or approximate timestamps.
- **First-Most Only:** Extract **only the first** timestamp boundary found. Ignore all subsequent data.
- **No Formatting:** Return **only** the numerical timestamp value (e.g., 50.707172). No extra text, no markdown, and no explanation.
- **Boundary Definition:** A shell prompt indicates a command cycle is complete. A simple switch between `<user_input>` and `<system_output>` is **NOT** a boundary.

### Example:
Input:
  <user_input timestamp="49.618611">\n</user_input>
  <system_output timestamp="50.17642">Removing libgail-common...</system_output>
  <system_output timestamp="50.707172">demo@faiserver:~$ </system_output>
  <user_input timestamp="53.176514">s</user_input>

The first-most boundary is found at the prompt in the third tag.
Target Timestamp: 50.707172

Return format:
50.707172
"""
xml_data = """
{"input": "<system_output timestamp=\"0.008584\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"0.603025\"/>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"0.603588\">\n(reverse-i-search)`': </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"0.937684\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"0.938181\">s': [7ms[27mync</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"1.114177\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"1.1146\">s': [7mss[27mh 172.16.0.17</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"1.291778\">h</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"1.292225\">[1@h': [7mssh[27m</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"2.617004\">\n</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"2.617465\">\n[8Pdemo@stephost:~$ ssh\n[?2004l\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"5.694771\">ssh: connect to host 172.16.0.17 port 22: No route to host\n\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"5.697578\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.138774\">c</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.139172\">c</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.367648\">m</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.368275\">m</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.491765\"> </user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.492239\"> </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.700208\">l</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.700707\">l</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"7.826627\">i</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"7.827056\">i</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.079826\">s</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.080258\">s</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.28657\">t</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.286967\">t</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"8.474631\">\n</user_input>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.475097\">\n[?2004l\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.48351\">implicitserver-tearoff-16: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.523754\">\nimplicitserver-tearoff-17: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.575367\">\nwikiserver-tearoff-3: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.604153\"> Running\nqemuwordserver-tearoff-4: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.63601\">\nlampserver-tearoff-14: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.667641\">\nwikiserver-tearoff-5: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.699481\">\ndrupalserver-tearoff-13: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.730786\">\nnullhost-tearoff-11: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.76265\">\nwikiserver-tearoff-2: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.794868\">\nfaiserver-tearoff-8: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.826481\">\nfaiserver-tearoff-9: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.858307\">\nfaiserver-tearoff-1: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.888755\">\nfaiserver-tearoff-19: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.918944\">\nnullhost-tearoff-10: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.952225\"> Running\nnullhost-tearoff-7: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"8.983719\">\nfaiserver-tearoff-18: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.015378\">\nfaiserver-tearoff-12: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.047348\">\nnullhost-tearoff-6: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.079908\"> Running\nnullhost-tearoff-15: Runable,</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.110634\">\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"9.110867\">[?2004hdemo@stephost:~$ </system_output>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<user_input timestamp=\"15.55465\"/>\n\n[Script Action: Processed XML chunk.]"}
{"input": "<system_output timestamp=\"15.555045\">[?2004l\n\nexit\n</system_output>\n\n[Script Action: Processed XML chunk.]"}
"""

In [20]:
# === Run inference (assumes `model` and `tokenizer` are already loaded) ===
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Extract event boundaries from this XML:\n{xml_data}"}
]

# Tokenize & move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate from model
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2,
)

import re

# raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]
raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# extract the assistant block using the marker tokens, if present
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
if m:
    raw_assistant = m.group(1).strip()
else:
    # fallback to full decode if markers missing
    raw_assistant = raw_output

print("=== RAW ASSISTANT OUTPUT ===")
print(raw_assistant)

# same extraction as above
timestamps = re.findall(r'\d+\.\d+', raw_assistant)
# dedupe-preserve-order
seen = set(); ordered = []
for t in timestamps:
    if t not in seen:
        seen.add(t); ordered.append(t)

print("\n=== CLEANED EXTRACTED TIMESTAMPS (assistant-only) ===")
print("\n".join(ordered) if ordered else "(no timestamps found)")


# === Print clearly separated input and model "true" output ===
print("\n=== CLEANED EXTRACTED TIMESTAMPS ===")
if timestamps:
    print("\n".join(timestamps))
else:
    print("(no timestamps found)")

=== RAW ASSISTANT OUTPUT ===
0.008584

=== CLEANED EXTRACTED TIMESTAMPS (assistant-only) ===
0.008584

=== CLEANED EXTRACTED TIMESTAMPS ===
0.008584


=============================================

Test the data cleanup feature.

In [22]:
raw_data = """
<system_output timestamp="67.122999"> all Help i18n of RFC822 compliant config files

ii iotop 0.6-2 i386 simple top-like I/O monitor

ii iproute2 4.9.0-1+deb9u1 i386 networking and traffic control tools

ii iptables 1.6.0+snapshot20161117-6 i386 administration tools for packet filtering and NAT

ii iputils-ping 3:20161105-1 i386 Tools to test the reachability of network hosts

ii isc-dhcp-client 4.3.5-3+deb9u1 i386 DHCP client for automatically obtaining an IP address

ii isc-dhcp-common 4.3.5-3+deb9u1 i386 common manpages relevant to all of the isc-dhcp packages

ii kbd 2.0.3-2+b1 i386 Linux console font and keytable utilities

ii keyutils 1.5.9-9 i386 Linux Key Management Utilities

ii klibc-utils 2.0.4-9 i386 small utilities built with klibc for early boot

ii kmod 23-2 i386 tools for managing Linux kernel modules

ii less 481-2.1 i386 pager program similar to more

ii libacl1:i386 2.2.52-3+b1 i386 Access control list shared library

ii libapparmor1:i386 2.11.0-3+deb9u2 i386 changehat AppArmor library

ii libapr1:i386 1.5.2-5 i386 Apache Portable R</system_output>

<system_output timestamp="67.124082">"
"""
SYSTEM_PROMPT = """
Use one sentence to conclude what is going on in this timestamp.
"""

In [23]:
# === Run inference (assumes `model` and `tokenizer` are already loaded) ===
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Extract event boundaries from this XML:\n{raw_data}"}
]

# Tokenize & move to GPU
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate from model
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.2,
)

raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]

# extract the assistant block using the marker tokens, if present
m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
if m:
    raw_assistant = m.group(1).strip()
else:
    # fallback to full decode if markers missing
    raw_assistant = raw_output

print("=== RAW ASSISTANT OUTPUT ===")
print(raw_assistant)

=== RAW ASSISTANT OUTPUT ===
At timestamp 67.122999, the system output lists various software packages and their versions installed on an i386 architecture, including tools for network monitoring, packet filtering, and system utilities.
